Lab 1 Solutions
Imports & Initialization

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DateType
from pyspark.sql.window import Window

# Initialize Spark Session (if not already running)
spark = SparkSession.builder.appName("SparkLab1").getOrCreate()

Section 1 — Basic DataFrame Operations

In [0]:
# Base Dataset [cite: 1, 2]
students = spark.createDataFrame([
    (101, "Ali",     "CS",      88),
    (102, "Mona",    "CS",      95),
    (103, "Youssef", "IT",      72),
    (104, "Nour",    "IT",      81),
    (105, "Sara",    "AI",      91),
    (106, "Omar",    "AI",      85)
], ["student_id", "name", "major", "score"])

# Exercise 1: Find all students in the AI major with scores greater than 90[cite: 2].
ex1_df = students.filter((F.col("major") == "AI") & (F.col("score") > 90))

# Exercise 2: Create a new column called result (Pass >= 80, Fail otherwise)[cite: 2].
ex2_df = students.withColumn(
    "result", 
    F.when(F.col("score") >= 80, "Pass").otherwise("Fail")
)

# Exercise 3: Calculate the average score per major sorted from highest to lowest[cite: 2].
ex3_df = students.groupBy("major") \
    .agg(F.avg("score").alias("avg_score")) \
    .sort(F.col("avg_score").desc())

# Exercise 4: Count the number of distinct majors[cite: 3].
distinct_majors_count = students.select("major").distinct().count()

# Exercise 5: Rename major to department and remove student_id[cite: 3].
ex5_df = students.withColumnRenamed("major", "department").drop("student_id")

# Display results
print("Exercise 1: AI major with score > 90")
display(ex1_df)

print("\nExercise 2: Students with Pass/Fail result")
display(ex2_df)

print("\nExercise 3: Average score per major")
display(ex3_df)

print("\nExercise 4: Count of distinct majors")
print(f"Distinct majors: {distinct_majors_count}")

print("\nExercise 5: Renamed and dropped columns")
display(ex5_df)

Exercise 1: AI major with score > 90


student_id,name,major,score
105,Sara,AI,91



Exercise 2: Students with Pass/Fail result


student_id,name,major,score,result
101,Ali,CS,88,Pass
102,Mona,CS,95,Pass
103,Youssef,IT,72,Fail
104,Nour,IT,81,Pass
105,Sara,AI,91,Pass
106,Omar,AI,85,Pass



Exercise 3: Average score per major


major,avg_score
CS,91.5
AI,88.0
IT,76.5



Exercise 4: Count of distinct majors
Distinct majors: 3

Exercise 5: Renamed and dropped columns


name,department,score
Ali,CS,88
Mona,CS,95
Youssef,IT,72
Nour,IT,81
Sara,AI,91
Omar,AI,85


Section 2 — Window Functions

In [0]:
# Exercise 1: Calculate the running total of scores ordered by student_id[cite: 4].
window_spec_running = Window.orderBy("student_id")
ex2_1_df = students.withColumn("running_total", F.sum("score").over(window_spec_running))

# Exercise 2: Create a column tagging above_major_avg / below_major_avg[cite: 5].
window_spec_major = Window.partitionBy("major")
ex2_2_df = students.withColumn("major_avg", F.avg("score").over(window_spec_major)) \
    .withColumn(
        "above_major_avg",
        F.when(F.col("score") > F.col("major_avg"), "above_major_avg")
         .otherwise("below_major_avg")
    ).drop("major_avg") # Cleaning up the temporary average column

# Exercise 3: Find the top 2 students from each major using a window function[cite: 6].
window_spec_rank = Window.partitionBy("major").orderBy(F.col("score").desc())
ex2_3_df = students.withColumn("rank", F.row_number().over(window_spec_rank)) \
    .filter(F.col("rank") <= 2) \
    .drop("rank")

# Display results
print("Exercise 1: Running total of scores")
display(ex2_1_df)

print("\nExercise 2: Above/below major average")
display(ex2_2_df)

print("\nExercise 3: Top 2 students per major")
display(ex2_3_df)

Exercise 1: Running total of scores


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


student_id,name,major,score,running_total
101,Ali,CS,88,88
102,Mona,CS,95,183
103,Youssef,IT,72,255
104,Nour,IT,81,336
105,Sara,AI,91,427
106,Omar,AI,85,512



Exercise 2: Above/below major average


student_id,name,major,score,above_major_avg
105,Sara,AI,91,above_major_avg
106,Omar,AI,85,below_major_avg
101,Ali,CS,88,below_major_avg
102,Mona,CS,95,above_major_avg
103,Youssef,IT,72,below_major_avg
104,Nour,IT,81,above_major_avg



Exercise 3: Top 2 students per major


student_id,name,major,score
105,Sara,AI,91
106,Omar,AI,85
102,Mona,CS,95
101,Ali,CS,88
104,Nour,IT,81
103,Youssef,IT,72


Section 3 — String Manipulation

In [0]:
# Base Dataset [cite: 7]
contacts = spark.createDataFrame([
    ("  ali123@example.com ",),
    ("sara_99@gmail.com",),
    ("mona.test@yahoo.com",),
    (" ahmed2025@outlook.com ",)
], ["email"])

# Exercise 3: Clean the email column first (helps optimize subsequent regex splits)[cite: 9].
cleaned_contacts = contacts.withColumn("email", F.lower(F.trim(F.col("email"))))

# Exercise 1 & 2: Extract username, email provider, and clean domain[cite: 7, 8].
# Regex breaks up everything before and after the '@' and separates the dot domain extension.
ex3_1_2_df = cleaned_contacts \
    .withColumn("username", F.regexp_extract(F.col("email"), r"^([^@]+)", 1)) \
    .withColumn("provider", F.regexp_extract(F.col("email"), r"@([^@]+)", 1)) \
    .withColumn("domain_name", F.regexp_extract(F.col("email"), r"@([^.]+)\.", 1))

# Exercise 4: Count how many emails belong to each provider[cite: 9].
ex3_4_df = ex3_1_2_df.groupBy("provider").count()

# Exercise 5: Replace all digits in usernames with an empty string[cite: 10].
ex3_5_df = ex3_1_2_df.withColumn("username_clean", F.regexp_replace(F.col("username"), r"\d", ""))

# Display results
print("Exercise 1 & 2: Extracted username, provider, and domain")
display(ex3_1_2_df)

print("\nExercise 4: Count emails per provider")
display(ex3_4_df)

print("\nExercise 5: Usernames with digits removed")
display(ex3_5_df)

Section 4 — Date Operations

In [0]:
# Base Dataset [cite: 11]
events = spark.createDataFrame([
    (1, "2025-01-15"),
    (2, "2025-03-22"),
    (3, "2025-07-04"),
    (4, "2025-10-19"),
    (5, "2025-12-31")
], ["event_id", "event_date"])

# Exercise 1 & 2 & 3: Convert to DateType, extract Year/Month/Day, and get Quarter[cite: 11, 12].
ex4_df = events \
    .withColumn("event_date", F.col("event_date").cast(DateType())) \
    .withColumn("year", F.year("event_date")) \
    .withColumn("month", F.month("event_date")) \
    .withColumn("day", F.dayofmonth("event_date")) \
    .withColumn("quarter", F.quarter("event_date"))

# Exercise 4: Calculate how many days remain until the end of the event's year[cite: 13].
ex4_df = ex4_df.withColumn(
    "days_remaining", 
    F.datediff(F.concat(F.col("year"), F.lit("-12-31")).cast(DateType()), F.col("event_date"))
)

# Exercise 5: Find all events that occurred in the second half of the year (July–December)[cite: 14].
ex4_5_df = ex4_df.filter(F.col("month") >= 7)

# Display results
print("Exercise 1-4: Date parsing and calculations")
display(ex4_df)

print("\nExercise 5: Events in second half of year")
display(ex4_5_df)

Section 5 — Complex Types (Arrays)

In [0]:
# Base Dataset [cite: 15]
courses = spark.createDataFrame([
    ("Python", ["Programming", "Beginner", "Data"]),
    ("Spark", ["Big Data", "Programming"]),
    ("SQL", ["Database", "Data"]),
    ("Airflow", ["Workflow", "Automation", "Data"]),
    ("Linux", ["OS", "Administration"])
], ["course_name", "topics"])

# Exercise 1 & 5: Find courses with > 2 topics and create a topic count column[cite: 15, 17].
ex5_1_5_df = courses.withColumn("topic_count", F.size(F.col("topics"))) \
                    .filter(F.col("topic_count") > 2)

# Exercise 2: Create a column has_data_topic (True/False)[cite: 16].
ex5_2_df = courses.withColumn("has_data_topic", F.array_contains(F.col("topics"), "Data"))

# Exercise 3: Use explode() and count how many courses contain each topic[cite: 16].
ex5_3_df = courses.withColumn("exploded_topic", F.explode(F.col("topics"))) \
                  .groupBy("exploded_topic") \
                  .count()

# Exercise 4: Find all courses that contain the topic "Programming"[cite: 17].
ex5_4_df = courses.filter(F.array_contains(F.col("topics"), "Programming"))

# Display results
print("Exercise 1 & 5: Courses with >2 topics and topic count")
display(ex5_1_5_df)

print("\nExercise 2: Courses with 'Data' topic")
display(ex5_2_df)

print("\nExercise 3: Topic occurrence count")
display(ex5_3_df)

print("\nExercise 4: Courses containing 'Programming'")
display(ex5_4_df)

Capstone Project — Online Learning Platform Analytics

In [0]:
# Base Datasets [cite: 18]
students_cap = spark.createDataFrame([
    (1, "Ali"), (2, "Mona"), (3, "Sara"), (4, "Omar"), (5, "Nour")
], ["student_id", "student_name"])

enrollments_cap = spark.createDataFrame([
    (1, "Python", 88), (1, "SQL", 92), (2, "Python", 95), (2, "Spark", 90),
    (3, "SQL", 78), (3, "Spark", 85), (4, "Python", 70), (5, "Spark", 98)
], ["student_id", "course", "score"])

# --- Part 1 — Data Preparation ---
# Task 1 & 3: Inner Join & Deduplication[cite: 18, 19].
joined_df = students_cap.join(enrollments_cap, on="student_id", how="inner").dropDuplicates()

# Task 2: Check schema and data quality[cite: 19].
print("--- Capstone Schema Validation ---")
joined_df.printSchema()

# --- Part 2 & 3 & 4 — Aggregations, Windowing & Business Rules ---
# Define Window specifications [cite: 22, 23]
course_window_desc = Window.partitionBy("course").orderBy(F.col("score").desc())
course_window_basic = Window.partitionBy("course")
course_window_running = Window.partitionBy("course").orderBy("score").rowsBetween(Window.unboundedPreceding, Window.currentRow)

final_analytics_df = joined_df \
    .withColumn("course_rank", F.rank().over(course_window_desc)) \
    .withColumn("course_running_total", F.sum("score").over(course_window_running)) \
    .withColumn("course_avg", F.avg("score").over(course_window_basic)) \
    .withColumn("score_diff_from_avg", F.col("score") - F.col("course_avg")) \
    .withColumn(
        "performance_category",
        F.when(F.col("score") >= 90, "Excellent")
         .when((F.col("score") >= 80) & (F.col("score") < 90), "Good")
         .otherwise("Needs Improvement")
    ) # [cite: 22, 23, 24]

# --- Part 5 — Final Report Generation --- [cite: 25]
report_df = final_analytics_df.select(
    "student_name", "course", "score", "course_rank", "performance_category"
).sort("course", "course_rank")

# Display Report [cite: 25]
print("--- Final Capstone Performance Report ---")
report_df.show()

# Extra KPIs requested in Part 2[cite: 20, 21]:
avg_score_per_student = joined_df.groupBy("student_name").agg(F.avg("score").alias("avg_score"))
avg_score_per_course = joined_df.groupBy("course").agg(F.avg("score").alias("avg_score"))
highest_score_per_course = joined_df.groupBy("course").agg(F.max("score").alias("max_score"))
highest_scoring_student = avg_score_per_student.sort(F.col("avg_score").desc()).limit(1)

# Display additional KPIs
print("\n--- Additional Analytics ---")
print("\nAverage score per student:")
display(avg_score_per_student)

print("\nAverage score per course:")
display(avg_score_per_course)

print("\nHighest score per course:")
display(highest_score_per_course)

print("\nHighest scoring student overall:")
display(highest_scoring_student)

Lab 2 Solutions
Section 1 — Basic Operations

In [0]:
# Base Dataset [cite: 26]
employees = spark.createDataFrame([
    (1, "Anna",    "Engineering", 95000),
    (2, "Ben",     "Marketing",   72000),
    (3, "Clara",   "Engineering", 88000),
    (4, "Dan",     "HR",          65000),
    (5, "Eva",     "Marketing",   79000),
    (6, "Frank",   "Engineering", 91000),
], ["id", "name", "dept", "salary"])

# Exercise 1: Filter Engineering employees with salary > 90,000[cite: 26, 27].
lab2_ex1 = employees.filter((F.col("dept") == "Engineering") & (F.col("salary") > 90000))

# Exercise 2: Salary evaluation column[cite: 27].
lab2_ex2 = employees.withColumn(
    "salary_grade", 
    F.when(F.col("salary") >= 80000, "High").otherwise("Standard")
)

# Exercise 3: Average salary per department sorted descending[cite: 28].
lab2_ex3 = employees.groupBy("dept").agg(F.avg("salary").alias("avg_salary")).sort(F.col("avg_salary").desc())

# Exercise 4: Count distinct departments[cite: 28].
distinct_dept_count = employees.select("dept").distinct().count()

# Exercise 5: Rename and drop columns[cite: 29].
lab2_ex5 = employees.withColumnRenamed("dept", "department").drop("id")

# Display results
print("Exercise 1: Engineering employees with salary > 90,000")
display(lab2_ex1)

print("\nExercise 2: Salary evaluation")
display(lab2_ex2)

print("\nExercise 3: Average salary per department")
display(lab2_ex3)

print("\nExercise 4: Distinct department count")
print(f"Distinct departments: {distinct_dept_count}")

print("\nExercise 5: Renamed and dropped columns")
display(lab2_ex5)

Section 2 — Window Functions

In [0]:
# Exercise 1: Running total of salary ordered by id[cite: 30].
emp_id_window = Window.orderBy("id")
lab2_sec2_ex1 = employees.withColumn("running_salary_total", F.sum("salary").over(emp_id_window))

# Exercise 2: Tag row based on department average[cite: 31].
emp_dept_window = Window.partitionBy("dept")
lab2_sec2_ex2 = employees.withColumn("dept_avg", F.avg("salary").over(emp_dept_window)) \
    .withColumn(
        "salary_tag",
        F.when(F.col("salary") > F.col("dept_avg"), "above_avg").otherwise("below_avg")
    ).drop("dept_avg")

# Exercise 3: Top 3 employees in Engineering by salary[cite: 32].
eng_salary_window = Window.partitionBy("dept").orderBy(F.col("salary").desc())
lab2_sec2_ex3 = employees.withColumn("salary_rank", F.row_number().over(eng_salary_window)) \
    .filter((F.col("dept") == "Engineering") & (F.col("salary_rank") <= 3)) \
    .drop("salary_rank")

# Display results
print("Exercise 1: Running total of salary")
display(lab2_sec2_ex1)

print("\nExercise 2: Above/below department average")
display(lab2_sec2_ex2)

print("\nExercise 3: Top 3 Engineering employees by salary")
display(lab2_sec2_ex3)

Section 3 — String Manipulations (Mock Log Data)

In [0]:
# Mock Data for testing Section 3 Exercises
logs_df = spark.createDataFrame([
    ("2026-03-15 10:14:00 INFO admin@domain.com",),
    ("2026-03-16 11:22:15 ERROR  user_99@service.org ",)
], ["log_line"])

# Exercise 1: Extract date and log level[cite: 33].
lab2_sec3_ex1 = logs_df \
    .withColumn("date", F.regexp_extract(F.col("log_line"), r"^(\d{4}-\d{2}-\d{2})", 1)) \
    .withColumn("log_level", F.regexp_extract(F.col("log_line"), r"\d{2}:\d{2}:\d{2}\s+([A-Z]+)", 1))

# Exercise 2: Given an email column, extract username and domain[cite: 34].
emails_mock = spark.createDataFrame([("test.user@company.com",)], ["email"])
lab2_sec3_ex2 = emails_mock \
    .withColumn("username", F.regexp_extract(F.col("email"), r"^([^@]+)", 1)) \
    .withColumn("domain", F.regexp_extract(F.col("email"), r"@([^@]+)$", 1))

# Exercise 3: Clean name column (trim, proper case, remove digits)[cite: 35].
names_mock = spark.createDataFrame([("  joHn123 dOe ",)], ["name"])
lab2_sec3_ex3 = names_mock \
    .withColumn("cleaned_name", F.initcap(F.trim(F.regexp_replace(F.col("name"), r"\d", ""))))

# Display results
print("Exercise 1: Extract date and log level from logs")
display(lab2_sec3_ex1)

print("\nExercise 2: Extract username and domain from email")
display(lab2_sec3_ex2)

print("\nExercise 3: Clean name column")
display(lab2_sec3_ex3)

Section 4 — Complex Dates (Mock Order Data)

In [0]:
# Mock Data for testing Section 4 Exercises
orders_mock = spark.createDataFrame([
    (1, "2026-02-14"), # Weekend (Saturday)
    (2, "2026-05-20")  # Weekday (Wednesday)
], ["order_id", "order_date"])

# Exercise 1: Format date parse, age evaluation, quarters, and end of month[cite: 36].
lab2_sec4_ex1 = orders_mock \
    .withColumn("order_date", F.col("order_date").cast(DateType())) \
    .withColumn("age_in_days", F.datediff(F.current_date(), F.col("order_date"))) \
    .withColumn("quarter", F.concat(F.lit("Q"), F.quarter("order_date"))) \
    .withColumn("last_day_of_month", F.last_day(F.col("order_date")))

# Exercise 2: Find all orders placed on a weekend (Saturday=6, Sunday=7 in PySpark default date_format 'u')[cite: 36].
lab2_sec4_ex2 = lab2_sec4_ex1.filter(
    F.date_format(F.col("order_date"), "u").isin(["6", "7"])
)

# Display results
print("Exercise 1: Date formatting, age, quarters, and end of month")
display(lab2_sec4_ex1)

print("\nExercise 2: Orders placed on weekends")
display(lab2_sec4_ex2)

Section 5 — Complex Array Manipulation

In [0]:
# Mock Data for testing Section 5 Exercises
products_mock = spark.createDataFrame([
    ("Laptop", ["Electronics", "Work", "Sale"]),
    ("Mug", ["Kitchen"]),
    ("Shoes", ["Apparel", "Sport", "Clearance", "Sale"])
], ["product_name", "promo_tags"])

# Exercise 1: Find all products that have more than 2 tags[cite: 37].
lab2_sec5_ex1 = products_mock.filter(F.size(F.col("promo_tags")) > 2)

# Exercise 2: Create a new column has_sale_tag (True/False)[cite: 38].
lab2_sec5_ex2 = products_mock.withColumn("has_sale_tag", F.array_contains(F.col("promo_tags"), "Sale"))

# Exercise 3: Use explode() + groupBy() to count tag occurrences[cite: 39].
lab2_sec5_ex3 = products_mock.withColumn("tag", F.explode(F.col("promo_tags"))) \
                             .groupBy("tag") \
                             .count()

# Display results
print("Exercise 1: Products with more than 2 tags")
display(lab2_sec5_ex1)

print("\nExercise 2: Products with 'Sale' tag")
display(lab2_sec5_ex2)

print("\nExercise 3: Tag occurrence count")
display(lab2_sec5_ex3)